###Data Reading JSON

In [0]:
df_json = spark.read.table('default.drivers')

In [0]:
df_json.display()

###Data Reading CSV



In [0]:
# Read from the existing big_mart_sales table instead of the DBFS path
df = spark.table('default.big_mart_sales')



In [0]:
df.show()

In [0]:
df.display()

### Print Schema

In [0]:
df.printSchema()

In [0]:
## Convert Item name to String 
my_ddlSchema='''
                Item_Identifier String,
                Item_Weight String,
                Item_Fat_Content String,
                Item_Visibility Double,
                Item_Type String,
                Item_MRP Double,
                Outlet_Identifier String,
                Outlet_Establishment_Year Long,
                Outlet_Size String,
                Outlet_Location_Type String,
                Outlet_Type String,
                Item_Outlet_Sales Double
                '''


In [0]:
# Apply the schema from my_ddlSchema using StructType.fromDDL()
from pyspark.sql.types import StructType

# Parse the DDL schema string
target_schema = StructType.fromDDL(my_ddlSchema)

# Read the table and apply the schema by casting each column
df = spark.table('default.big_mart_sales')
for field in target_schema.fields:
    df = df.withColumn(field.name, df[field.name].cast(field.dataType))

df.show()
df.display()
df.printSchema()

In [0]:
df.printSchema()

### Structype Schema()

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
my_strct_schema=StructType([
                                StructField('Item_Identfier',StringType(),True),
                                StructField('Item_Weight',StringType(),True),
                                StructField('Item_Fat_Content',StringType(),True),
                                StructField('Item_Visibility',DoubleType(),True),
                                StructField('Item_Type',StringType(),True),
                                StructField('Item_MRP',DoubleType(),True),
                                StructField('Outlet_Identifier',StringType(),True),
                                StructField('Outlet_Establishment_Year',IntegerType(),True),
                                StructField('Outlet_Size',StringType(),True),
                                StructField('Outlet_Location_Type',StringType(),True),
                                StructField('Outlet_Type',StringType(),True),
                                StructField('Item_Outlet_Sales',DoubleType(),True)

])

In [0]:
# Reload df from table to ensure it's properly initialized
df = spark.table('default.big_mart_sales')
df.printSchema()

In [0]:
df = spark.table('default.big_mart_sales')

###SELECT

In [0]:
df.display()

In [0]:
df_sel=df.select('Item_Identifier','Item_Weight','Item_Identifier').display()

ALIAS

In [0]:
from pyspark.sql.functions import col
df.select(col('Item_Identifier').alias('ItemId')).display()

###FILTER/WHERE

#### Scenerio 1

In [0]:
df.display()

In [0]:
df.filter(col('Item_Fat_Content')=='Regular').display()

#### Scenerio 2

In [0]:
df.filter((col('Item_Type') =='Soft Drinks') & (col('Item_Weight')<10)).display()

#### Scenerio 3

In [0]:
df.filter((col('Outlet_Size').isNull()) & (col('Outlet_Location_Type').isin(['Tier 1','Tier 2']))).display()

### withcolumnrename()

In [0]:
df.withColumnRenamed('Item_Weight','Weight').display()

### withColumn

new_col

In [0]:
df=df.withColumn('flag',lit("new")).display()

In [0]:
df = spark.table('default.big_mart_sales')
df.withColumn('multiply',col('Item_Weight')*col('Item_MRP')).display()

In [0]:
df.withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),'LF','Low Fat'))\
    .withColumn('Item_fat_Content',regexp_replace(col('Item_Fat_Content'),'Regular','Regular Fat')).display()

### Type Casting

In [0]:
df=df.withColumn('Item_Weight',col('Item_Weight').cast(StringType()))

In [0]:
df.printSchema()

###Sort/Order By


In [0]:
df.sort(col('Item_Weight').desc()).display()

In [0]:
df.sort(col('Item_Visibility').asc()).display()

In [0]:
df.sort('Item_Weight','Item_Visibility',ascending=[0,0]).display()

In [0]:
df.sort(['Item_Weight','Item_Visibility'],acsending=[1,0]).display()

###LIMIT

In [0]:
df.limit(10).display()

### DROP

In [0]:
df.drop('Item_Visibility').display()

In [0]:
df.drop('Item_Visibility','Item_Weight').display()

### DROP DUPLICATES

In [0]:
df.dropDuplicates().display()

In [0]:
df.dropDuplicates(subset=['Item_Type']).display()

In [0]:
df.distinct().display()

### Union and Union Byname

#### Preparing Data Frames


In [0]:
data_1 = [(1,'kad'),(2,'sid')]
schema1 = 'id String, name String'

df1=spark.createDataFrame(data_1,schema1)

data_2 = [(3,'rahul'),(4,'sid')]
schema2 = 'id String, name String'

df2=spark.createDataFrame(data_2,schema2)


In [0]:
df1.display()

In [0]:
df2.display()

In [0]:
df1.union(df2).display()

In [0]:
data_1 = [('kad',1),('sid',2)]
schema1 = 'name String, id String'

df1=spark.createDataFrame(data_1,schema1)
df1.union(df2).display()

In [0]:
df1.unionByName(df2).display()

### Init cap, lower case, upper case

In [0]:
from pyspark.sql.functions import *
df.select(initcap('Item_Type')).display()

In [0]:
df.select(lower('Item_Type')).display()

In [0]:
df.select(upper('Item_Type')).display()

### CurrentDate, Date add, Date sub

In [0]:
df=df.withColumn('curr_date',current_date())
df.display()

In [0]:
df=df.withColumn('week after',date_add('curr_date',7)).display()

In [0]:
df = spark.table('default.big_mart_sales')
df = df.withColumn('curr_date', current_date())
df=df.withColumn('week before',date_sub('curr_date',7))
df.display()
#df.withColumn('week before',date_add('curr_date',-7)).display()

In [0]:
df=df.withColumn('datediff',datediff('curr_date','week before'))
df.display()

In [0]:
# Reload df to clear any cached transformations from previous cell
df = spark.table('default.big_mart_sales')
df = df.withColumn('curr_date', current_date())
df = df.withColumn('week before', date_sub('curr_date', 7))
df = df.withColumn('datediff', datediff('curr_date', 'week before'))
df = df.withColumn('week before', date_format('week before', 'dd-MM-yyyy'))
df.display()

### Handling NULL VALUES

In [0]:
df.dropna('all').display()

In [0]:
df.dropna('any').display()

In [0]:
df.dropna(subset=['Outlet_Size']).display()

### Filling Nulls


In [0]:
df.fillna('Not_Availale').display()

In [0]:
df.fillna('Not_Avaialble',subset=['Outlet_Size']).display()

### Split and indexing

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')).display()

In [0]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')[1]).display()

### Explode

In [0]:
df_extra=df.withColumn('Outlet_Type',split('Outlet_type',' '))
df_extra.display()

In [0]:
df_extra.withColumn('Outlet_Type',explode('Outlet_Type')).display()

### Array Contains

In [0]:
df_extra.withColumn('Type1Flag',array_contains('Outlet_Type','Type1')).display()

### Group By

In [0]:
df.groupBy('Item_Type').agg(sum("Item_MRP")).display()

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP').alias("Totol MRP")).display()

In [0]:
df = spark.table('default.big_mart_sales')

In [0]:
from pyspark.sql.functions import *
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP'),avg('Item_MRP')).display()

### Collect_List

In [0]:
df.groupBy('Item_Type').agg(collect_list('Outlet_Establishment_Year')).display()

### Pivot

In [0]:
df.groupBy('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).display()

### When Otherwise

In [0]:
df.withColumn('Veg_Flag',when(col("Item_Type")=='Meat','Non-Veg').otherwise('Veg')).display()

In [0]:
df.withColumn('Veg_Exp_Flag',when((col("Item_Type") != 'Meat') & (col("Item_MRP")<100),"Veg_Inexpensive")\
    .when((col("Item_Type") != 'Meat') & (col("Item_MRP")>100),"Veg_Expensive").otherwise('NA')).display()

### Joins

In [0]:
dataj1 = [('1', 'gaur', 'd01'),
        ('2', 'kit', 'd02'),
        ('3','sam', 'd03'),
        ('4', 'tim', 'd03'),
        ('5','aman', 'd05')]

schemaj1='emp_id STRING, emp_name STRING, dept_id STRING'

df1= spark.createDataFrame(dataj1, schemaj1)

dataj2 = [('d01', 'HR'),
        ('d02', 'Marketing'),
        ('d03', 'Accounts'),
        ('d04', 'IT'),
        ('d05', 'Finance')]

schemaj2= 'dept_id STRING, department STRING'

df2= spark.createDataFrame (dataj2, schemaj2)

In [0]:
df1.display()

In [0]:
df2.display()

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'inner').display()

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'left').display()

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'right').display()

In [0]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'anti').display()

### Widnow Functions

In [0]:
from pyspark.sql.window import Window

In [0]:
from pyspark.sql.functions import row_number
df.withColumn('rowCol',row_number().over(Window.orderBy('Item_Identifier'))).display()

### Rank and Dense Rank

In [0]:
from pyspark.sql.functions import *
df.withColumn('rank', rank().over(Window.partitionBy(col('Item_Identifier')).orderBy(col('Item_Outlet_Sales')))).display()

In [0]:
rank_window = Window.partitionBy("Item_Identifier").orderBy("Item_Outlet_Sales")

dense_rank_window = Window.orderBy(desc("Item_Outlet_Sales"))

df = df.withColumn(
    "rank",
    rank().over(rank_window)
).withColumn(
    "dense_rank",
    dense_rank().over(dense_rank_window)
)

df.display()

#### Cumulative Sum

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_Type'))).display()

In [0]:
df = spark.table('default.big_mart_sales')

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))).display()

### User Defined Functions (UDF)

In [0]:
def my_fun(x):
    return x*x
print(my_fun(3))

In [0]:
my_udf=udf(my_fun)

In [0]:
df.withColumn('mynewcol',my_udf(col('Item_MRP'))).display()


### Data Writing

In [0]:
df.write.saveAsTable('workspace.default.data_csv')

 #### Modes- Append, Overwrite, Error, Ignore

In [0]:
df.write.mode('append').saveAsTable('workspace.default.data_csv')

In [0]:
df.write.mode('overwrite').saveAsTable('workspace.default.data_csv')

In [0]:
df.write.mode('error').saveAsTable('workspace.default.data_csv')

In [0]:
df.write.mode('ignore').saveAsTable('workspace.default.data_csv')

### Parquet File Format- It is stored as row based file format and it is very handy when we wish to write data means in OLTP databases or transactional databases but in case of big data we cannot use this file format as it used as columnar. Metadata is stored at the footer of the file

In [0]:
spark.sql("SHOW CATALOGS").show(truncate=False)

In [0]:
spark.sql("SHOW SCHEMAS").show(truncate=False)

In [0]:
df.write.mode("overwrite") \
    .parquet("/Volumes/workspace/default/myvolume/data_csv")

### Data File Format - built over parquet file. Metadata is stored in seperate file (delta log) not in the same file

###Managed vs External Tables


SPARK SQL- createTempView

In [0]:
df.createTempView('my_view')

In [0]:
%sql
select * from my_view

In [0]:
%sql
select * from my_view where Item_fat_Content='LF'

In [0]:
df_new=spark.sql("select * from my_view where Item_fat_Content='LF'")
df_new.display()